In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import skew, kurtosis

from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from statsmodels.regression.rolling import RollingOLS
import statsmodels.api as sm

print("All imports done.")

In [ ]:
raw = pd.read_excel(
    "../data/rt_sem2_data.xlsx",
    index_col=0,
    parse_dates=True
)

if isinstance(raw.columns, pd.MultiIndex):
    raw.columns = raw.columns.get_level_values(-1)

raw = raw.rename(columns={'^VIX': 'VIX', '^TNX': 'TNX', '^IRX': 'IRX'})
raw = raw.ffill().dropna(how='all')

print(raw.shape)
print(raw.columns.tolist())

In [ ]:
# --- Log Returns ---
def compute_log_returns(df, assets):
    returns = pd.DataFrame(index=df.index)
    for asset in assets:
        returns[asset] = np.log(df[asset] / df[asset].shift(1))
    return returns.dropna()


# --- Rolling Statistical Features ---
def rolling_features(returns, windows=[21, 63]):
    features = pd.DataFrame(index=returns.index)
    for asset in returns.columns:
        for w in windows:
            features[f'{asset}_vol_{w}d'] = returns[asset].rolling(w).std()
            features[f'{asset}_skew_{w}d'] = returns[asset].rolling(w).apply(lambda x: skew(x), raw=True)
            features[f'{asset}_kurt_{w}d'] = returns[asset].rolling(w).apply(lambda x: kurtosis(x), raw=True)

            def entropy(x):
                hist, _ = np.histogram(x, bins=10, density=True)
                hist = hist[hist > 0]
                return -np.sum(hist * np.log(hist))

            features[f'{asset}_entropy_{w}d'] = returns[asset].rolling(w).apply(entropy, raw=True)
    return features


# --- Hurst Exponent (on PRICES, as per paper) ---
def hurst_rs_single(x):
    x = np.array(x, dtype=float)
    if len(x) < 10:
        return np.nan
    mean_adj = x - np.mean(x)
    z = np.cumsum(mean_adj)
    r = np.max(z) - np.min(z)
    s = np.std(x, ddof=0)
    if s == 0:
        return np.nan
    return np.log(r / s) / np.log(len(x))

def hurst_features(prices):
    features = pd.DataFrame(index=prices.index)
    windows = {'short': 64, 'medium': 128, 'long': 256}
    for asset in prices.columns:
        for name, window in windows.items():
            features[f'{asset}_hurst_{name}'] = prices[asset].rolling(window).apply(
                hurst_rs_single, raw=True
            )
    return features


# --- Cross Asset Features (Beta + Correlation vs SPY) ---
def cross_asset_features(returns, target='SPY', windows=[21, 63]):
    features = pd.DataFrame(index=returns.index)
    spy_r = returns[target]
    for asset in returns.columns:
        if asset == target:
            continue
        for w in windows:
            spy_var = spy_r.rolling(w).var()
            cov = returns[asset].rolling(w).cov(spy_r)
            features[f'{asset}_beta_{target}_{w}d'] = cov / spy_var
            features[f'{asset}_corr_{target}_{w}d'] = returns[asset].rolling(w).corr(spy_r)
    return features


# --- KL Divergence ---
def rolling_kl(series, w_curr=21, w_ref=126, bins=30):
    values = series.values.astype(np.float64)
    n = len(values)
    result = np.full(n, np.nan)
    for t in range(max(w_curr, w_ref), n):
        curr = values[t - w_curr + 1: t + 1]
        ref  = values[t - w_ref + 1:  t + 1]
        curr = curr[~np.isnan(curr)]
        ref  = ref[~np.isnan(ref)]
        if len(curr) < 8 or len(ref) < 8:
            continue
        vmin = min(curr.min(), ref.min())
        vmax = max(curr.max(), ref.max())
        if vmax - vmin < 1e-12:
            result[t] = 0.0
            continue
        edges = np.linspace(vmin, vmax, bins + 1)
        p_curr, _ = np.histogram(curr, bins=edges, density=True)
        p_ref,  _ = np.histogram(ref,  bins=edges, density=True)
        p_curr = p_curr + 1e-10
        p_ref  = p_ref  + 1e-10
        result[t] = np.sum(p_curr * np.log(p_curr / p_ref))
    return pd.Series(result, index=series.index)

def kl_features(returns):
    features = pd.DataFrame(index=returns.index)
    for asset in returns.columns:
        features[f'{asset}_kl'] = rolling_kl(returns[asset])
    return features


# --- Master Build Function ---
def build_features(df, assets):
    returns = compute_log_returns(df, assets)
    stats   = rolling_features(returns)
    hurst   = hurst_features(df[assets])          # PRICES, not returns
    cross   = cross_asset_features(returns)
    kl      = kl_features(returns)

    all_features = pd.concat([stats, hurst, cross, kl], axis=1)
    all_features = all_features.ffill()

    scaler = StandardScaler()
    features_scaled = pd.DataFrame(
        scaler.fit_transform(all_features),
        index=all_features.index,
        columns=all_features.columns
    )
    return features_scaled, returns

print("All functions defined.")

In [ ]:
assets = raw.columns.tolist()
features, returns = build_features(raw, assets)

def create_target(returns, horizon=5, threshold=-0.01):
    future_returns = returns['SPY'].rolling(horizon).sum().shift(-horizon)
    return (future_returns <= threshold).astype(int)

y = create_target(returns)

# Align features and target
df_combined = features.copy()
df_combined['target'] = y
df_combined = df_combined.dropna()

X = df_combined.drop(columns=['target'])
y = df_combined['target']

print(f"X shape: {X.shape}")
print(f"Target distribution:\n{y.value_counts()}")

In [ ]:
# Step 1: Low variance removal
variances = X.var()
X = X[variances[variances > 1e-4].index]
print(f"After low variance removal: {X.shape[1]} features")

# Step 2: High correlation removal
corr_matrix = X.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.95)]
X = X.drop(columns=to_drop)
print(f"After correlation removal: {X.shape[1]} features")

# Step 3: Mutual Information - top 80
mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({'feature': X.columns, 'score': mi_scores}).sort_values('score', ascending=False)
selected_features = mi_df.head(80)['feature'].tolist()
X_selected = X[selected_features]

print(f"Final selected features: {X_selected.shape[1]}")
print(f"\nTop 13 features by MI:\n{mi_df.head(13).to_string(index=False)}")

In [ ]:
plt.figure(figsize=(12, 4))
plt.bar(range(len(mi_df)), mi_df['score'].values, color='steelblue')
plt.xlabel('Feature Rank')
plt.ylabel('MI with Target')
plt.title('Mutual Information Scores (Top-to-Bottom Order)')
plt.tight_layout()
plt.show()

In [ ]:
# Paper uses full dataset for in-sample training
X_train = X_selected.copy()
y_train = y.copy()

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)

print(f"Training on {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Crash rate: {y_train.mean():.2%}")

In [ ]:
print("Training MLP...")
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=300,
    random_state=42
)
mlp.fit(X_train_scaled, y_train)

print("Training XGBoost...")
xgb = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    eval_metric='logloss',
    random_state=42,
    verbosity=0
)
xgb.fit(X_train, y_train)

print("Training CatBoost...")
cat = CatBoostClassifier(
    iterations=200,
    depth=5,
    learning_rate=0.05,
    verbose=0,
    random_state=42
)
cat.fit(X_train, y_train)

print("All models trained.")

In [ ]:
p_mlp = mlp.predict_proba(X_train_scaled)[:, 1]
p_xgb = xgb.predict_proba(X_train)[:, 1]
p_cat = cat.predict_proba(X_train)[:, 1]

p_final = (p_mlp + p_xgb + p_cat) / 3
y_pred  = (p_final >= 0.5).astype(int)

print(f"Accuracy  : {accuracy_score(y_train, y_pred):.4f}")
print(f"Precision : {precision_score(y_train, y_pred, zero_division=0):.4f}")
print(f"Recall    : {recall_score(y_train, y_pred, zero_division=0):.4f}")
print(f"F1 Score  : {f1_score(y_train, y_pred, zero_division=0):.4f}")
print(f"ROC AUC   : {roc_auc_score(y_train, p_final):.4f}")

print("\nConfusion Matrix:")
cm = confusion_matrix(y_train, y_pred)
print(cm)

print("\nClassification Report:")
print(classification_report(y_train, y_pred, zero_division=0))

In [ ]:
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predicted 0', 'Predicted 1'],
            yticklabels=['Actual 0', 'Actual 1'])
plt.title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(y_train, p_final)
auc_score = roc_auc_score(y_train, p_final)

plt.figure(figsize=(8, 5))
plt.plot(fpr, tpr, color='blue', label=f'Ensemble AUC = {auc_score:.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('In-Sample ROC Curve')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
#SHAP starts from here

In [ ]:
#pip install shap

In [ ]:
# Run this if you haven't installed shap yet
# !pip install shap

import shap
shap.initjs()
print("SHAP ready.")

In [ ]:
# We run SHAP on XGBoost — TreeExplainer is exact and fast for tree models
# The paper applies SHAP to the ensemble; we use XGBoost as the representative
# tree model since TreeExplainer doesn't work on averaged ensembles directly

explainer = shap.TreeExplainer(xgb)
shap_values = explainer.shap_values(X_train)

# shap_values shape: (n_samples, n_features)
# Positive value = pushed prediction toward crash (class 1)
# Negative value = pushed prediction toward no-crash (class 0)

print(f"SHAP values shape: {shap_values.shape}")
print("SHAP computation done.")

In [ ]:
#Top 10 SHAP Features During Crash Weeks
# Split into crash and non-crash weeks (matching paper Section 3.2)
crash_idx    = (y_train == 1).values
noncrash_idx = (y_train == 0).values

shap_crash    = shap_values[crash_idx]
shap_noncrash = shap_values[noncrash_idx]

# Mean absolute SHAP per feature for crash weeks
mean_shap_crash = pd.Series(
    np.abs(shap_crash).mean(axis=0),
    index=X_train.columns
).sort_values(ascending=False).head(10)

# Plot — matches Figure 3 in paper
plt.figure(figsize=(10, 6))
plt.barh(mean_shap_crash.index[::-1], mean_shap_crash.values[::-1], color='steelblue')
plt.xlabel('Mean |SHAP value|')
plt.title("Top 10 SHAP Features for class 'Crash'")
plt.tight_layout()
plt.show()

print("\nTop 10 features driving CRASH predictions:")
print(mean_shap_crash.round(5))

In [ ]:
# Top 10 SHAP Features During Non-Crash Weeks
# Mean absolute SHAP per feature for non-crash weeks
mean_shap_noncrash = pd.Series(
    np.abs(shap_noncrash).mean(axis=0),
    index=X_train.columns
).sort_values(ascending=False).head(10)

# Plot — matches Figure 4 in paper
plt.figure(figsize=(10, 6))
plt.barh(mean_shap_noncrash.index[::-1], mean_shap_noncrash.values[::-1], color='lightcoral')
plt.xlabel('Mean |SHAP value|')
plt.title("Top 10 SHAP Features for class 'No Crash'")
plt.tight_layout()
plt.show()

print("\nTop 10 features driving NON-CRASH predictions:")
print(mean_shap_noncrash.round(5))

In [ ]:
# Paper's key finding: Hurst dominated MI but disappears in SHAP
# Let's verify this in our results

# Get Hurst features from top MI list
top_mi_features = mi_df.head(20)['feature'].tolist()
hurst_in_top_mi = [f for f in top_mi_features if 'hurst' in f]

# Get Hurst features from top SHAP list
top_shap_crash_features = mean_shap_crash.index.tolist()
hurst_in_top_shap = [f for f in top_shap_crash_features if 'hurst' in f]

print("=== Paper's Key Finding: MI vs SHAP Divergence ===")
print(f"\nHurst features in Top 20 by MI    : {len(hurst_in_top_mi)}")
print(f"  → {hurst_in_top_mi}")
print(f"\nHurst features in Top 10 SHAP (crash): {len(hurst_in_top_shap)}")
print(f"  → {hurst_in_top_shap}")
print("\nInterpretation: MI captures unconditional dependence.")
print("SHAP captures conditional contribution given all other features.")
print("If Hurst drops out of SHAP, it means cross-asset signals absorb its information.")

In [ ]:
# Pick the most recent observation — this is what your dashboard will show live
# This shows exactly WHY the model predicted what it did on that specific day

latest_idx = -1  # last row
sample = X_train.iloc[[latest_idx]]

explanation = explainer(sample)

plt.figure(figsize=(10, 6))
shap.waterfall_plot(explanation[0], max_display=10, show=False)
plt.title(f"SHAP Waterfall — Most Recent Observation ({X_train.index[latest_idx].date()})")
plt.tight_layout()
plt.show()

print(f"\nModel's crash probability for this day: {p_final[latest_idx]:.2%}")
print(f"Actual outcome: {'CRASH' if y_train.iloc[latest_idx] == 1 else 'NO CRASH'}")

In [ ]:
# Signed mean SHAP — positive means pushes toward crash, negative means away from crash
signed_crash    = pd.Series(shap_crash.mean(axis=0),    index=X_train.columns)
signed_noncrash = pd.Series(shap_noncrash.mean(axis=0), index=X_train.columns)

# Top 10 features for crash weeks with their direction
top_crash_signed = signed_crash.abs().sort_values(ascending=False).head(10)
top_noncrash_signed = signed_noncrash.abs().sort_values(ascending=False).head(10)

# Build comparison table
all_top = list(set(top_crash_signed.index.tolist() + top_noncrash_signed.index.tolist()))

comparison = pd.DataFrame({
    'SHAP_in_CrashWeeks'   : signed_crash[all_top].round(5),
    'SHAP_in_NonCrashWeeks': signed_noncrash[all_top].round(5),
}, index=all_top).sort_values('SHAP_in_CrashWeeks', ascending=False)

print("=== Signed SHAP: Direction of each feature in each regime ===")
print("Positive = pushes toward CRASH prediction")
print("Negative = pushes toward NO CRASH prediction")
print()
print(comparison.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Crash weeks
top10_crash = signed_crash.abs().sort_values(ascending=False).head(10)
colors_crash = ['red' if signed_crash[f] > 0 else 'blue' for f in top10_crash.index]
axes[0].barh(top10_crash.index[::-1], top10_crash.values[::-1], color=colors_crash[::-1])
axes[0].set_xlabel('Mean |SHAP value|')
axes[0].set_title("Top 10 SHAP Features — Crash Weeks\n(red=increases crash risk, blue=reduces it)")

# Non-crash weeks  
top10_noncrash = signed_noncrash.abs().sort_values(ascending=False).head(10)
colors_noncrash = ['red' if signed_noncrash[f] > 0 else 'blue' for f in top10_noncrash.index]
axes[1].barh(top10_noncrash.index[::-1], top10_noncrash.values[::-1], color=colors_noncrash[::-1])
axes[1].set_xlabel('Mean |SHAP value|')
axes[1].set_title("Top 10 SHAP Features — Non-Crash Weeks\n(red=increases crash risk, blue=reduces it)")

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# RISK QUINTILE ANALYSIS — Section 3.3 of the paper
# Goal: Show that weeks our model flagged as HIGH risk actually
# had worse SPY returns, and LOW risk weeks had better returns.
# This proves the model has real economic meaning, not just
# good classification numbers on paper.
# ================================================================

# Step 1: Get the actual SPY forward returns for each observation
# These are the REAL returns that happened after each prediction
# We already computed this in create_target() — let's rebuild it cleanly

spy_log_returns = returns['SPY']  # daily log returns of SPY

# Rolling 5-day forward return (same horizon as our target)
# shift(-5) means: "what happened in the NEXT 5 days from this date"
spy_5day_forward = spy_log_returns.rolling(5).sum().shift(-5)
spy_5day_forward.name = 'spy_5day_return'

# Step 2: Align with our predicted probabilities
# p_final contains the model's crash probability for each training observation
# X_train.index gives us the dates those predictions correspond to

prob_series = pd.Series(p_final, index=X_train.index, name='crash_prob')

# Combine predictions + actual forward returns into one dataframe
quintile_df = pd.concat([prob_series, spy_5day_forward], axis=1).dropna()

print(f"Observations available for quintile analysis: {len(quintile_df)}")
print(f"Date range: {quintile_df.index[0].date()} → {quintile_df.index[-1].date()}")

In [ ]:
# ================================================================
# Step 3: Split all observations into 5 equal buckets (quintiles)
# based on predicted crash probability
#
# Q0 = lowest 20% crash probability → model says VERY SAFE
# Q1 = next 20%
# Q2 = middle 20%
# Q3 = next 20%
# Q4 = highest 20% crash probability → model says VERY RISKY
#
# If the model is good, Q0 should have positive SPY returns
# and Q4 should have negative SPY returns — a clean monotonic pattern
# This is exactly Figure 5 in the paper
# ================================================================

quintile_df['quintile'] = pd.qcut(
    quintile_df['crash_prob'],
    q=5,
    labels=[0, 1, 2, 3, 4]   # 0=lowest risk, 4=highest risk
)

# Step 4: Compute average 5-day SPY return within each quintile
avg_return_by_quintile = quintile_df.groupby('quintile')['spy_5day_return'].mean()

print("Average 5-day SPY return by predicted risk quintile:")
print("(Negative = model correctly identified risky periods)")
print()
for q, ret in avg_return_by_quintile.items():
    label = "← LOWEST RISK" if q == 0 else ("← HIGHEST RISK" if q == 4 else "")
    print(f"  Q{q}: {ret*100:+.3f}%  {label}")

In [ ]:
# ================================================================
# Plot: Average SPY Return per Quintile
# This is Figure 5 in the paper
# A clean downward pattern (Q0 positive → Q4 negative) confirms
# the model captures true conditional risk, not statistical noise
# ================================================================

fig, ax = plt.subplots(figsize=(9, 5))

colors = ['green' if r > 0 else 'red'
          for r in avg_return_by_quintile.values]

bars = ax.bar(
    [f'Q{i}' for i in range(5)],
    avg_return_by_quintile.values * 100,   # convert to percentage
    color=colors,
    edgecolor='black',
    linewidth=0.5
)

# Add value labels on top of each bar
for bar, val in zip(bars, avg_return_by_quintile.values * 100):
    ypos = val + 0.02 if val >= 0 else val - 0.05
    ax.text(bar.get_x() + bar.get_width() / 2,
            ypos, f'{val:+.3f}%',
            ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Predicted Risk Quintile  (Q0 = Lowest Risk → Q4 = Highest Risk)', fontsize=11)
ax.set_ylabel('Average 5-Day SPY Return (%)', fontsize=11)
ax.set_title('Average SPY Return by Predicted Crash Probability Quintile\n'
             '(Replicating Figure 5 from the paper)', fontsize=12)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Paper's simplification: collapse quintiles into just two groups
# LOW risk  = Q0 to Q3 (bottom 80% of crash probability)
# HIGH risk = Q4       (top 20% of crash probability)
#
# Then compare average returns and plot the distribution
# The paper found:
#   HIGH risk average = -1.89%
#   LOW  risk average = +0.75%
# Let's see what our model produces
# ================================================================

# Label each observation as HIGH or LOW risk
quintile_df['risk_regime'] = quintile_df['quintile'].apply(
    lambda q: 'HIGH Risk' if q == 4 else 'LOW Risk'
)

# Compute averages for both groups
avg_high = quintile_df[quintile_df['risk_regime'] == 'HIGH Risk']['spy_5day_return'].mean()
avg_low  = quintile_df[quintile_df['risk_regime'] == 'LOW Risk']['spy_5day_return'].mean()

print("=" * 45)
print("HIGH vs LOW Risk Period — Average 5-Day SPY Return")
print("=" * 45)
print(f"  HIGH Risk periods (Q4): {avg_high*100:+.3f}%")
print(f"  LOW  Risk periods (Q0-Q3): {avg_low*100:+.3f}%")
print(f"  Difference: {(avg_high - avg_low)*100:.3f}%")
print()
print(f"  Paper reported: HIGH = -1.89%,  LOW = +0.75%")
print(f"  Your results  : HIGH = {avg_high*100:+.3f}%, LOW = {avg_low*100:+.3f}%")

In [ ]:
# ================================================================
# Distribution plot: shows the SPREAD of returns in each regime
# Key things to look for:
#   - HIGH risk should have a fat left tail (big negative returns)
#   - LOW  risk should be tight and centered near zero or positive
# This matches Figure 6 in the paper
# ================================================================

fig, ax = plt.subplots(figsize=(10, 5))

high_returns = quintile_df[quintile_df['risk_regime'] == 'HIGH Risk']['spy_5day_return'] * 100
low_returns  = quintile_df[quintile_df['risk_regime'] == 'LOW Risk']['spy_5day_return'] * 100

ax.hist(high_returns, bins=60, alpha=0.6, color='red',
        label=f'HIGH Risk (Q4)  avg={avg_high*100:+.2f}%', density=False)
ax.hist(low_returns,  bins=60, alpha=0.5, color='green',
        label=f'LOW Risk (Q0-Q3) avg={avg_low*100:+.2f}%', density=False)

ax.axvline(avg_high * 100, color='darkred',   linestyle='--', linewidth=1.5)
ax.axvline(avg_low  * 100, color='darkgreen', linestyle='--', linewidth=1.5)
ax.axvline(0, color='black', linestyle='-', linewidth=0.8)

ax.set_xlabel('5-Day SPY Return (%)', fontsize=11)
ax.set_ylabel('Frequency', fontsize=11)
ax.set_title('Forward 5-Day SPY Returns: HIGH Risk vs LOW Risk Periods\n'
             '(Replicating Figure 6 from the paper)', fontsize=12)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Final summary statistics for each quintile
# Gives a complete picture: not just average return but also
# how often the model was right, and how bad the worst weeks were
# ================================================================

summary = quintile_df.groupby('quintile')['spy_5day_return'].agg([
    ('Avg Return %',  lambda x: round(x.mean() * 100, 3)),
    ('Std Dev %',     lambda x: round(x.std() * 100, 3)),
    ('Min Return %',  lambda x: round(x.min() * 100, 3)),
    ('Max Return %',  lambda x: round(x.max() * 100, 3)),
    ('Count',         'count')
])

# Add average crash probability per quintile
summary['Avg Crash Prob'] = quintile_df.groupby('quintile')['crash_prob'].mean().round(3)

print("=" * 70)
print("Full Quintile Summary Table")
print("=" * 70)
print(summary.to_string())
print()
print("Key insight: If 'Avg Return %' decreases monotonically from Q0 to Q4,")
print("the model has genuine predictive power over realized market returns.")

In [ ]:
print("End of Quantile Analysis")

trading strategy

In [ ]:
# ================================================================
# TRADING STRATEGY — Section 4 of the paper
#
# Core logic (Section 4, bullet points):
# - If model's crash probability < 0.5 → LONG SPY (position = +1)
# - If model's crash probability >= 0.5 → SHORT SPY (position = -1)
#
# Position sizing (Section 4, bullet 3):
# The paper scales position by predicted probability so that
# high confidence signals get bigger positions.
# Formula: position = 1 - 2 * crash_prob
#   crash_prob = 0.1 → position = +0.8 (confident long)
#   crash_prob = 0.5 → position =  0.0 (neutral)
#   crash_prob = 0.9 → position = -0.8 (confident short)
#
# Your daily return = position × what SPY actually did that day
# ================================================================

# Step 1: Create a series of positions for every day in our dataset
prob_series = pd.Series(p_final, index=X_train.index, name='crash_prob')

# Scale position by probability (paper Section 4 — position sizing)
# Values range from +1 (very safe) to -1 (very risky)
positions = 1 - 2 * prob_series
positions.name = 'position'

# Step 2: Get SPY's actual daily log return for each of those days
spy_daily = returns['SPY']

# Step 3: Align positions with SPY returns (same dates only)
aligned = pd.concat([positions, spy_daily], axis=1).dropna()
aligned.columns = ['position', 'spy_return']

# Step 4: Strategy daily return = position × SPY return
# If you were long (+0.8) and SPY went up 1% → you made +0.8%
# If you were short (-0.8) and SPY went down 1% → you made +0.8%
aligned['strategy_return'] = aligned['position'] * aligned['spy_return']

print(f"Strategy period: {aligned.index[0].date()} → {aligned.index[-1].date()}")
print(f"Total trading days: {len(aligned)}")
print(f"\nSample of first 5 rows:")
print(aligned.head().round(4))

didnt understand, do again

In [ ]:
# ================================================================
# Compute the exact metrics from Table 4 in the paper:
# Sharpe Ratio, Max Drawdown, Annualized Return,
# Annualized Volatility, CAPM Alpha, CAPM Beta, T-stat
# ================================================================
import scipy.stats as stats

strat_ret = aligned['strategy_return']
spy_ret   = aligned['spy_return']

# --- Annualized Return ---
# Compound daily returns to yearly figure
# 252 = trading days in a year
annualized_return = (1 + strat_ret.mean()) ** 252 - 1

# --- Annualized Volatility ---
# How much the strategy fluctuates per year
annualized_vol = strat_ret.std() * np.sqrt(252)

# --- Sharpe Ratio ---
# Return per unit of risk (using 0% risk-free rate)
# Paper uses same assumption
sharpe = (strat_ret.mean() / strat_ret.std()) * np.sqrt(252)

# --- Maximum Drawdown ---
# Worst peak-to-trough loss the strategy ever suffered
cumulative = (1 + strat_ret).cumprod()  # portfolio value over time
rolling_max = cumulative.cummax()        # track the peak at each point
drawdown = (cumulative - rolling_max) / rolling_max  # how far below peak
max_drawdown = drawdown.min()            # worst point

# --- CAPM Alpha and Beta ---
# Simple linear regression: strategy_return = alpha + beta * spy_return
# alpha = intercept (excess return beyond market)
# beta  = slope (how much you move with the market)
# This is literally just a straight line fit — no complex finance needed
X_capm = sm.add_constant(spy_ret)  # adds intercept column
capm_model = sm.OLS(strat_ret, X_capm).fit()

alpha_daily = capm_model.params['const']   # daily alpha
beta        = capm_model.params['spy_return']  # market sensitivity
alpha_annual = alpha_daily * 252           # annualize it
t_stat_alpha = capm_model.tvalues['const'] # how statistically significant

# --- Information Ratio vs SPY ---
# How consistently did you beat SPY?
# Higher = more consistent outperformance
active_return = strat_ret - spy_ret
info_ratio = (active_return.mean() / active_return.std()) * np.sqrt(252)

# --- Print results table (matching Table 4 in paper) ---
print("=" * 50)
print("   TRADING STRATEGY PERFORMANCE (Table 4)")
print("=" * 50)
print(f"  Sharpe Ratio            : {sharpe:.4f}   (paper: 2.51)")
print(f"  Information Ratio vs SPY: {info_ratio:.4f}   (paper: 1.73)")
print(f"  Maximum Drawdown        : {max_drawdown*100:.2f}%  (paper: -18.12%)")
print(f"  Annualized Return       : {annualized_return*100:.2f}%  (paper: 40.84%)")
print(f"  Annualized Volatility   : {annualized_vol*100:.2f}%  (paper: 13.23%)")
print(f"  CAPM Alpha (daily)      : {alpha_daily:.5f}   (paper: 0.00111)")
print(f"  CAPM Alpha (annualized) : {alpha_annual:.4f}   (paper: 0.28)")
print(f"  CAPM Beta               : {beta:.4f}   (paper: 0.51)")
print(f"  T-stat Alpha            : {t_stat_alpha:.4f}   (paper: 14.03)")
print("=" * 50)
print("\nNote: In-sample results. No transaction costs.")
print("Paper explicitly flags this limitation (Section 4.2).")

In [ ]:
# ================================================================
# Plot strategy cumulative returns vs just holding SPY
# This is the "equity curve" — shows how your portfolio grew
# over time compared to the benchmark
# A good strategy should:
# 1. Grow faster than SPY overall
# 2. Fall less than SPY during crashes (2008, 2020, 2022)
# ================================================================

# Cumulative returns (start both at 100)
strat_cumulative = (1 + strat_ret).cumprod() * 100
spy_cumulative   = (1 + spy_ret).cumprod() * 100

fig, axes = plt.subplots(2, 1, figsize=(13, 10))

# --- Top plot: Cumulative returns over time ---
axes[0].plot(strat_cumulative.index, strat_cumulative.values,
             color='green', linewidth=1.5, label='Strategy')
axes[0].plot(spy_cumulative.index, spy_cumulative.values,
             color='blue', linewidth=1.5, label='SPY Buy & Hold', alpha=0.7)

axes[0].set_title('Cumulative Returns: Strategy vs SPY Buy & Hold\n'
                  '(In-sample, 2005–2025)', fontsize=12)
axes[0].set_ylabel('Portfolio Value (Base = 100)', fontsize=10)
axes[0].legend(fontsize=10)
axes[0].grid(alpha=0.3)

# Add annotations for key market events
events = {
    '2008\nGFC':   '2008-09-15',
    '2020\nCOVID': '2020-03-23',
    '2022\nRate\nHikes': '2022-01-03'
}
for label, date in events.items():
    try:
        x = pd.Timestamp(date)
        if x in strat_cumulative.index:
            axes[0].axvline(x, color='gray', linestyle=':', alpha=0.7)
            axes[0].text(x, strat_cumulative.max() * 0.6,
                        label, fontsize=8, color='gray', ha='center')
    except:
        pass

# --- Bottom plot: Drawdown over time ---
axes[1].fill_between(drawdown.index, drawdown.values * 100, 0,
                     color='red', alpha=0.4, label='Strategy Drawdown')
axes[1].set_title('Strategy Drawdown Over Time', fontsize=12)
axes[1].set_ylabel('Drawdown (%)', fontsize=10)
axes[1].set_xlabel('Date', fontsize=10)
axes[1].legend(fontsize=10)
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ================================================================
# Compare the distribution of daily returns:
# Strategy vs SPY
#
# What to look for (paper Section 4.1):
# 1. Strategy should have a THINNER left tail than SPY
#    (fewer extreme negative days = better downside protection)
# 2. Strategy right tail should be similar to SPY
#    (you still participate in good days)
# 3. Strategy has a spike near zero
#    (days when position is near-neutral = no P&L)
# ================================================================

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(spy_ret * 100, bins=100, alpha=0.5, color='blue',
        label='SPY Daily Returns', density=True)
ax.hist(strat_ret * 100, bins=100, alpha=0.6, color='green',
        label='Strategy Daily Returns', density=True)

ax.set_xlabel('Daily Return (%)', fontsize=11)
ax.set_ylabel('Density', fontsize=11)
ax.set_title('Distribution of Daily Returns: Strategy vs SPY\n'
             '(Strategy should have thinner left tail — Figure 8 equivalent)',
             fontsize=12)
ax.legend(fontsize=10)
ax.set_xlim(-12, 12)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ================================================================
# Show the trading signal alongside SPY price
# Green triangles = model said BUY (low crash risk)
# Red triangles = model said SELL/SHORT (high crash risk)
# This is Figure 7 in the paper
# Shows visually that the model correctly flagged major downturns
# ================================================================

# Use last 4 years for clarity (too crowded over 20 years)
recent = aligned['2021':]
spy_price = raw['SPY']['2021':]

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(spy_price.index, spy_price.values,
        color='black', linewidth=1, label='SPY Price', alpha=0.8)

# Long signals (crash prob < 0.5)
long_dates  = recent[recent['position'] > 0].index
short_dates = recent[recent['position'] < 0].index

ax.scatter(long_dates,
           spy_price.reindex(long_dates),
           marker='^', color='green', s=15, alpha=0.6, label='Long (Low Risk)')
ax.scatter(short_dates,
           spy_price.reindex(short_dates),
           marker='v', color='red', s=15, alpha=0.6, label='Short (High Risk)')

ax.set_title('SPY Price with Model Trading Signals (2021–2025)\n'
             'Replicating Figure 7 from the paper', fontsize=12)
ax.set_ylabel('SPY Price', fontsize=10)
ax.set_xlabel('Date', fontsize=10)
ax.legend(fontsize=10)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()